In [1]:
!pip install sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentele

In [2]:
documents = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks.",
    "Python is a popular programming language.",
    "Transformers are powerful NLP models.",
    "ChromaDB is used for vector storage."
]

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
import chromadb

client = chromadb.Client()
collection = client.get_or_create_collection(name="documents")

In [5]:
embeddings = model.encode(documents)

ids = [str(i) for i in range(len(documents))]

collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=ids
)

In [6]:
query = "AI models"

query_embedding = model.encode([query])[0]

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

for i, doc in enumerate(results["documents"][0]):
    print(f"{i+1}. {doc}")

1. Machine learning is a subset of artificial intelligence.
2. Deep learning uses neural networks.
3. Transformers are powerful NLP models.


### Experiment 1: Adding more documents with metadata

Let's add some more documents to our collection, this time including associated metadata. Metadata can be useful for filtering results during queries.

In [7]:
new_documents = [
    "Generative AI models can create text, images, and other media.",
    "The Python ecosystem has a rich set of libraries for data science.",
    "Natural Language Processing (NLP) deals with the interaction between computers and human language.",
    "Vector databases are essential for efficient similarity search in AI applications.",
    "PyTorch and TensorFlow are popular deep learning frameworks."
]

new_metadatas = [
    {"source": "AI", "topic": "Generative AI"},
    {"source": "Programming", "topic": "Python"},
    {"source": "AI", "topic": "NLP"},
    {"source": "Database", "topic": "Vector Databases"},
]

# Generate new IDs for these documents
start_id = len(documents) # Using the length of the previous documents list
new_ids = [str(i + start_id) for i in range(len(new_documents))]

# Encode new documents
new_embeddings = model.encode(new_documents)

# Add new documents with embeddings and metadata to the collection
collection.add(
    documents=new_documents,
    embeddings=new_embeddings,
    metadatas=new_metadatas,
    ids=new_ids
)

print(f"Added {len(new_documents)} new documents to the collection.")
print(f"Total documents in collection: {collection.count()}")

Added 5 new documents to the collection.
Total documents in collection: 10


### Experiment 2: Querying with different `n_results`

Let's see how changing `n_results` affects the number of returned documents.

In [8]:
query = "artificial intelligence"
query_embedding = model.encode([query])[0]

print(f"\nQuery: '{query}'")

print("\n--- Results with n_results=1 ---")
results_1 = collection.query(
    query_embeddings=[query_embedding],
    n_results=1
)
for i, doc in enumerate(results_1["documents"][0]):
    print(f"{i+1}. {doc}")

print("\n--- Results with n_results=5 ---")
results_5 = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)
for i, doc in enumerate(results_5["documents"][0]):
    print(f"{i+1}. {doc}")


Query: 'artificial intelligence'

--- Results with n_results=1 ---
1. Machine learning is a subset of artificial intelligence.

--- Results with n_results=5 ---
1. Machine learning is a subset of artificial intelligence.
2. Generative AI models can create text, images, and other media.
3. Deep learning uses neural networks.
4. Natural Language Processing (NLP) deals with the interaction between computers and human language.
5. PyTorch and TensorFlow are popular deep learning frameworks.


### Experiment 3: Querying with metadata filters

Now, let's query for documents related to 'programming' that also have a specific 'topic' or 'source' in their metadata.

In [9]:
query = "programming languages and tools"
query_embedding = model.encode([query])[0]

print(f"\nQuery: '{query}'")

print("\n--- Results filtered by topic='Python' ---")
results_filtered_python = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    where={"topic": "Python"}
)
for i, doc in enumerate(results_filtered_python["documents"][0]):
    print(f"{i+1}. {doc}")

print("\n--- Results filtered by source='AI' ---")
results_filtered_ai = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    where={"source": "AI"}
)
for i, doc in enumerate(results_filtered_ai["documents"][0]):
    print(f"{i+1}. {doc}")

print("\n--- Results filtered by source='Database' ---")
results_filtered_db = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    where={"source": "Database"}
)
for i, doc in enumerate(results_filtered_db["documents"][0]):
    print(f"{i+1}. {doc}")


Query: 'programming languages and tools'

--- Results filtered by topic='Python' ---
1. The Python ecosystem has a rich set of libraries for data science.

--- Results filtered by source='AI' ---
1. Natural Language Processing (NLP) deals with the interaction between computers and human language.
2. PyTorch and TensorFlow are popular deep learning frameworks.
3. Generative AI models can create text, images, and other media.

--- Results filtered by source='Database' ---
1. Vector databases are essential for efficient similarity search in AI applications.


### Experiment 4: Querying with multiple metadata filters

Let's combine multiple conditions in the `where` clause to get more specific results.

In [10]:
query = "AI and programming"
query_embedding = model.encode([query])[0]

print(f"\nQuery: '{query}'")

print("\n--- Results filtered by source='AI' AND topic='Deep Learning' ---")
results_filtered_ai_dl = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    where={"$and": [{"source": "AI"}, {"topic": "Deep Learning"}]}
)
for i, doc in enumerate(results_filtered_ai_dl["documents"][0]):
    print(f"{i+1}. {doc}")

print("\n--- Results filtered by source='AI' OR topic='Python' ---")
results_filtered_ai_or_python = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    where={"$or": [{"source": "AI"}, {"topic": "Python"}]}
)
for i, doc in enumerate(results_filtered_ai_or_python["documents"][0]):
    print(f"{i+1}. {doc}")


Query: 'AI and programming'

--- Results filtered by source='AI' AND topic='Deep Learning' ---
1. PyTorch and TensorFlow are popular deep learning frameworks.

--- Results filtered by source='AI' OR topic='Python' ---
1. Generative AI models can create text, images, and other media.
2. Natural Language Processing (NLP) deals with the interaction between computers and human language.
3. PyTorch and TensorFlow are popular deep learning frameworks.


### Experiment 5: Deleting documents

We can remove documents from the collection by their IDs. Let's delete a couple of the newly added documents and check the collection count.

In [11]:
print(f"Total documents before deletion: {collection.count()}")

# Let's delete documents with IDs '5' and '6'
delete_ids = ['5', '6']
collection.delete(ids=delete_ids)

print(f"Deleted documents with IDs: {delete_ids}")
print(f"Total documents after deletion: {collection.count()}")

# Try querying for one of the deleted documents' content to confirm it's gone
print("\n--- Querying for a deleted document (Generative AI models...) ---")
query_deleted_doc = "Generative AI models can create text, images, and other media."
query_deleted_embedding = model.encode([query_deleted_doc])[0]
results_after_delete = collection.query(
    query_embeddings=[query_deleted_embedding],
    n_results=1
)
# If the document is truly deleted, this query should return a different, less relevant document
if results_after_delete["documents"] and results_after_delete["documents"][0]:
    print(f"Closest document after deletion: {results_after_delete['documents'][0][0]}")
else:
    print("No documents found for the query after deletion, as expected if it was deleted.")

Total documents before deletion: 10
Deleted documents with IDs: ['5', '6']
Total documents after deletion: 8

--- Querying for a deleted document (Generative AI models...) ---
Closest document after deletion: Transformers are powerful NLP models.


### Experiment 6: Querying with `where_document` filter

This filter allows us to search within the content of the documents themselves, using operators like `$contains`.

In [12]:
# Define the query string for the similarity search.
query = "learning"
# Encode the query string into a vector embedding using the pre-trained model.
query_embedding = model.encode([query])[0]

# Print the query for context.
print(f"\nQuery: '{query}'")

# --- First query with where_document filter for 'deep' ---
print("\n--- Results filtered by document content containing 'deep' ---")
# Query the collection for documents that are semantically similar to the query embedding
# AND whose content contains the substring 'deep'.
results_where_doc_deep = collection.query(
    query_embeddings=[query_embedding],
    n_results=3, # Request up to 3 results
    where_document={"$contains": "deep"} # Filter condition: document content must contain "deep"
)
# Iterate through the returned documents and print them.
for i, doc in enumerate(results_where_doc_deep["documents"][0]):
    print(f"{i+1}. {doc}")

# --- Second query with where_document filter for 'python' ---
print("\n--- Results filtered by document content containing 'python' ---")
# Query the collection again, this time for documents semantically similar to the query embedding
# AND whose content contains the substring 'python'.
results_where_doc_python = collection.query(
    query_embeddings=[query_embedding],
    n_results=3, # Request up to 3 results
    where_document={"$contains": "python"} # Filter condition: document content must contain "python"
)
# Iterate through the returned documents and print them.
for i, doc in enumerate(results_where_doc_python["documents"][0]):
    print(f"{i+1}. {doc}")


Query: 'learning'

--- Results filtered by document content containing 'deep' ---
1. PyTorch and TensorFlow are popular deep learning frameworks.

--- Results filtered by document content containing 'python' ---


### Experiment 7: Updating existing document metadata

Let's update the metadata for an existing document. We'll pick a document by its ID and change its associated metadata.

In [13]:
# Let's choose a document to update. We'll pick the first document we added (ID '0').
document_id_to_update = '0'

# First, let's retrieve its current metadata to see what we're changing (optional).
current_data = collection.get(ids=[document_id_to_update], include=['metadatas'])
print(f"Current metadata for ID '{document_id_to_update}': {current_data['metadatas'][0]}")

# Define the new metadata.
# Note: When updating, you provide the full new metadata object for the specified ID.
# ChromaDB doesn't merge metadata; it replaces it for the given ID.
new_metadata = {"source": "AI_Fundamentals", "difficulty": "Beginner"}

# Update the document's metadata
collection.update(
    ids=[document_id_to_update],
    metadatas=[new_metadata]
)

print(f"\nUpdated metadata for ID '{document_id_to_update}' to: {new_metadata}")

# Retrieve the document again to verify the update.
updated_data = collection.get(ids=[document_id_to_update], include=['metadatas'])
print(f"Verified metadata for ID '{document_id_to_update}': {updated_data['metadatas'][0]}")

# Also, let's try querying using the new metadata to ensure it's effective.
query_text = "artificial intelligence basics"
query_embedding = model.encode([query_text])[0]

print("\n--- Querying with new metadata filter (source='AI_Fundamentals') ---")
results_filtered_updated_meta = collection.query(
    query_embeddings=[query_embedding],
    n_results=1,
    where={"$and": [{"source": "AI_Fundamentals"}, {"difficulty": "Beginner"}]}
)

if results_filtered_updated_meta["documents"] and results_filtered_updated_meta["documents"][0]:
    print(f"Found document: {results_filtered_updated_meta['documents'][0][0]}")
    print(f"With metadata: {results_filtered_updated_meta['metadatas'][0][0]}")
else:
    print("No document found matching the query with the new metadata filter.")

Current metadata for ID '0': None

Updated metadata for ID '0' to: {'source': 'AI_Fundamentals', 'difficulty': 'Beginner'}
Verified metadata for ID '0': {'difficulty': 'Beginner', 'source': 'AI_Fundamentals'}

--- Querying with new metadata filter (source='AI_Fundamentals') ---
Found document: Machine learning is a subset of artificial intelligence.
With metadata: {'source': 'AI_Fundamentals', 'difficulty': 'Beginner'}


In [14]:
query = "learning"
query_embedding = model.encode([query])[0]

print(f"\nQuery: '{query}'")

print("\n--- Results filtered by document content containing 'deep' ---")
results_where_doc_deep = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    where_document={"$contains": "deep"}
)
for i, doc in enumerate(results_where_doc_deep["documents"][0]):
    print(f"{i+1}. {doc}")

print("\n--- Results filtered by document content containing 'python' ---")
results_where_doc_python = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    where_document={"$contains": "python"}
)
for i, doc in enumerate(results_where_doc_python["documents"][0]):
    print(f"{i+1}. {doc}")


Query: 'learning'

--- Results filtered by document content containing 'deep' ---
1. PyTorch and TensorFlow are popular deep learning frameworks.

--- Results filtered by document content containing 'python' ---


### Experiment 8: Bulk Update Multiple Documents

Let's perform a bulk update on the metadata of a few documents and verify the changes.

In [15]:
# Let's choose a couple of documents to update by their IDs
# We'll pick ID '1' and ID '2' (originally 'Deep learning uses neural networks.' and 'Python is a popular programming language.')
# Note: ID '5' and '6' were deleted earlier, so we'll pick from existing ones.
update_ids = ['1', '2']

# Define the new metadatas for these documents. The order must match `update_ids`.
new_bulk_metadatas = [
    {"source": "AI_Specialized", "level": "Advanced"},
]

# --- Before Update ---
print("\n--- Current Metadatas Before Bulk Update ---")
current_bulk_data = collection.get(ids=update_ids, include=['metadatas'])
for i, doc_id in enumerate(update_ids):
    print(f"ID '{doc_id}': {current_bulk_data['metadatas'][i]}")

# Perform the bulk update
collection.update(
    ids=update_ids,
    metadatas=new_bulk_metadatas
)
print("\nPerformed bulk update.")

# --- After Update ---
print("\n--- Verified Metadatas After Bulk Update ---")
updated_bulk_data = collection.get(ids=update_ids, include=['metadatas'])
for i, doc_id in enumerate(update_ids):
    print(f"ID '{doc_id}': {updated_bulk_data['metadatas'][i]}")

# You can also add some queries to confirm the new metadata filters work
print("\n--- Querying with new bulk-updated metadata filter (source='AI_Specialized') ---")
bulk_query_text = "advanced AI concepts"
bulk_query_embedding = model.encode([bulk_query_text])[0]
results_bulk_filtered = collection.query(
    query_embeddings=[bulk_query_embedding],
    n_results=1,
    where={"source": "AI_Specialized"}
)
if results_bulk_filtered["documents"] and results_bulk_filtered["documents"][0]:
    print(f"Found document: {results_bulk_filtered['documents'][0][0]}")
    print(f"With metadata: {results_bulk_filtered['metadatas'][0][0]}")
else:
    print("No document found matching the bulk-updated query filter.")


--- Current Metadatas Before Bulk Update ---
ID '1': None
ID '2': None

Performed bulk update.

--- Verified Metadatas After Bulk Update ---
ID '1': {'source': 'AI_Specialized', 'level': 'Advanced'}
ID '2': {'language': 'Python', 'source': 'Programming_Fundamentals'}

--- Querying with new bulk-updated metadata filter (source='AI_Specialized') ---
Found document: Deep learning uses neural networks.
With metadata: {'level': 'Advanced', 'source': 'AI_Specialized'}
